# 🚕 What Kinds of RoboTaxi Crashes Are There?
### K-Means Clustering of Autonomous Vehicle Crashes
**PROG8431 · Problem Analysis Workshop B · Team 3**

| Name | Student ID |
| :--- | :--- |
| Sultan Atanda | 9114837 |
| Ricardo Mohammed | 7500382 |
| Alamir Ibrahim | 9104960 |

> **Data:** NHTSA Standing General Order 2021-01 crash reports (cleaned in our earlier workshops)<br>
> **Question:** when a RoboTaxi is *moving* and crashes, do the crashes fall into natural groups?<br>
> **Repo Link:** https://github.com/kinghighshow/WorkshopB_Team_3.ipynb.git

**Roadmap:** ① Refine the data → ② Pick 2 variables → ③ Examine & standardize → ④ Run K-Means → ⑤ Choose k → ⑥ Name & explain the clusters

# Environment Setup

In [ ]:
%pip install -q pandas numpy scikit-learn plotly altair

### Start by clearing all output and workspace

In [ ]:
from IPython.display import clear_output
clear_output()
%reset -f

### Import necessary packages and set constants

In [ ]:
import os
import urllib.request
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
import altair as alt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

DATA_URL = "https://raw.githubusercontent.com/kinghighshow/WorkshopB_Team_3.ipynb/main/autonomous_vehicle_incidents_cleaned.csv"
LOCAL_FILE = "autonomous_vehicle_incidents_cleaned.csv"

SPEED = "SV Precrash Speed (MPH)"
YEAR = "Model Year"
COLS = ["State", "City", "Roadway Type", "Crash With", "SV Pre-Crash Movement", "CP Pre-Crash Movement"]
MAX_K = 7
K_RANGE = range(1, MAX_K + 1)
FINAL_K = 3
SEED = 123

PALETTE = ["#2563EB", "#F97316", "#10B981", "#A855F7", "#EF4444", "#EAB308", "#14B8A6"]
MUTED = "#94A3B8"
INK = "#0F172A"
pio.templates["team3"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=15, color=INK),
    title=dict(font=dict(size=21), x=0.01), margin=dict(l=70, r=30, t=90, b=60),
    colorway=PALETTE))
pio.templates.default = "plotly_white+team3"

# 📂 Import the data
The cleaned crash file from our previous workshops (1,257 crashes, 13 columns). It is downloaded from our [GitHub repo](https://github.com/kinghighshow/WorkshopB_Team_3.ipynb) the first time, then read from disk.

In [ ]:
if not os.path.exists(LOCAL_FILE):
    urllib.request.urlretrieve(DATA_URL, LOCAL_FILE)

crashes = pd.read_csv(LOCAL_FILE)
missing = {SPEED, YEAR, *COLS} - set(crashes.columns)
assert not missing, f"Missing columns: {missing}"

crashes[SPEED] = pd.to_numeric(crashes[SPEED], errors="coerce")
crashes[YEAR] = pd.to_numeric(crashes[YEAR], errors="coerce")
crashes = crashes.dropna(subset=[SPEED, YEAR])

print(f"{len(crashes):,} crashes · {crashes.shape[1]} columns")
crashes.head()

1,257 crashes · 14 columns


,Make,Model Year,Automation System Engaged?,City,State,Roadway Type,Crash With,CP Pre-Crash Movement,SV Pre-Crash Movement,SV Precrash Speed (MPH),Within ODD?,Roadway_Conditions,Weather_Conditions,Crash_Interaction_Type
0,JAGUAR,2021,ADS,San Francisco,CA,Intersection,Passenger Car,Entering Traffic,Proceeding Straight,7,Yes,No Unusual Conditions,Clear,Motor Vehicle
1,HYUNDAI,2026,ADS,Dallas,TX,Street,SUV,Changing Lanes,Proceeding Straight,15,Yes,No Unusual Conditions,Clear,Motor Vehicle
2,HYUNDAI,2026,ADS,Dallas,TX,Street,Passenger Car,Stopped,Proceeding Straight,4,Yes,Traffic Incident,Partly Cloudy,Motor Vehicle
3,HYUNDAI,2026,ADS,Dallas,TX,Intersection,SUV,Traveling Wrong Way,Proceeding Straight,9,Yes,No Unusual Conditions,Clear,Motor Vehicle
4,HYUNDAI,2025,ADS,Austin,TX,Intersection,SUV,Entering Traffic,Proceeding Straight,21,Yes,No Unusual Conditions,Clear,Motor Vehicle


## ① Refine the data: remove the 0 mph crashes

In [ ]:
stopped = crashes[crashes[SPEED] == 0]
moving = crashes[crashes[SPEED] > 0].copy().reset_index(drop=True)

fig = go.Figure(go.Bar(
    x=[len(stopped), len(moving)], y=["Stopped (0 mph)", "Moving (> 0 mph)"], orientation="h",
    marker_color=[MUTED, PALETTE[0]],
    text=[f"{len(stopped):,} crashes · {len(stopped)/len(crashes):.0%}",
          f"{len(moving):,} crashes · {len(moving)/len(crashes):.0%}"],
    textposition="inside", textfont=dict(size=17, color="white")))
fig.update_layout(height=300, title="<b>More than half of all crashes happened while the RoboTaxi was stopped</b>",
                  xaxis_title="Number of crashes")
fig.show()

print("What the stopped RoboTaxi was doing:", stopped["SV Pre-Crash Movement"].value_counts().head(3).to_dict())

What the stopped RoboTaxi was doing: {'Stopped': 565, 'Parked': 109, 'Proceeding Straight': 3}


**Why we filtered out the zeros (our instructor's suggestion):**
- **55% of crashes are at 0 mph**: the RoboTaxi was **stopped or parked** and another road user hit it.
- These crashes all have the **same speed value**, so they would pile up into one giant blob and drown out every other pattern.
- They tell us about *other drivers*, not about how the **RoboTaxi drives**. Our question is about crashes while the RoboTaxi is moving.
- **Data quirk:** a handful of 0 mph crashes are coded as *Proceeding Straight*. A vehicle creeping forward can round down to 0 mph, so we keep them with the stopped group, since speed is what defines it.
- ➜ We keep the **567 moving crashes** for clustering.

## ② How clustering helps us, and which 2 variables we chose

**How clustering deepens our understanding**
- In Workshop A we compared groups **we chose** (Motor Vehicle vs Fixed Object).
- Clustering lets the **data choose the groups**. K-Means finds crashes that are similar to each other, without us telling it what to look for.
- We then use **all the other columns** (road type, crash partner, state, city) to explain what each group means.

**Our two variables (the "centres")**

| Variable | Why we chose it |
| :--- | :--- |
| `SV Precrash Speed (MPH)` | Speed is the best measure of **how serious** a crash can be. It's our main variable since Workshop A. |
| `Model Year` | Tells us the **vehicle generation**. Newer RoboTaxis have newer software and sensors, so crashes may differ by generation. |

These are the **only two numeric columns** in our data. All the others are categories (state, city, weather), which K-Means cannot measure distance on, so we use them to *describe* the clusters instead.

**A limitation we are aware of:** `Model Year` only takes 8 distinct values and most moving crashes involve 2024 vehicles, so it behaves almost like a category. We keep it because it is the only other numeric signal, and we read its cluster with that in mind.

## ③ Examine and standardize the variables

### Examine the variables
Before clustering, we look at how our two variables are distributed.
- **Speed** is a continuous measure, so a **box plot** suits it.
- **Model Year** only has a few distinct values. A box plot of it collapses into a flat line, because the 25th percentile, the median and the 75th percentile are **all 2024**. A **bar chart of counts per year** shows its shape honestly.

In [ ]:
# @title
year_counts = moving[YEAR].value_counts().sort_index()
top_year = year_counts.idxmax()
top_share = year_counts.max() / len(moving)

fig = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], horizontal_spacing=0.12,
                    subplot_titles=("Speed is right-skewed",
                                    f"{top_share:.0%} of moving crashes involve {top_year} vehicles"))
fig.add_trace(go.Box(y=moving[SPEED], name="Speed", marker_color=PALETTE[0], boxmean=True,
                     boxpoints="outliers", line_width=2), row=1, col=1)
fig.add_trace(go.Bar(x=year_counts.index.astype(str), y=year_counts.values, text=year_counts.values,
                     textposition="outside", cliponaxis=False,
                     marker_color=[PALETTE[1] if y == top_year else MUTED for y in year_counts.index]),
              row=1, col=2)

fig.update_layout(height=480, showlegend=False, title="<b>Our two clustering variables (moving crashes only)</b>")
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text="Speed (mph)", row=1, col=1)
fig.update_xaxes(title_text="Model year", type="category", row=1, col=2)
fig.update_yaxes(title_text="Number of crashes", range=[0, year_counts.max() * 1.15], row=1, col=2)
fig.show()

q = moving[YEAR].quantile([0.25, 0.5, 0.75]).astype(int).tolist()
print(f"Speed: median {moving[SPEED].median():.0f} mph, 75% of crashes at or below {moving[SPEED].quantile(0.75):.0f} mph, max {moving[SPEED].max():.0f} mph")
print(f"Model year quartiles (Q1, median, Q3): {q}, which is why a box plot of it is a flat line")

Speed: median 9 mph, 75% of crashes at or below 18 mph, max 65 mph
Model year quartiles (Q1, median, Q3): [2024, 2024, 2024], which is why a box plot of it is a flat line


**What we see**
- **Speed:** strongly right-skewed. Half of the moving crashes happen at **9 mph or less** and 75% at **18 mph or less**, with a long tail up to **65 mph**.
- **Model Year:** heavily concentrated on **2024** (about 71% of moving crashes), with a thin tail of older 2017 to 2022 vehicles and some 2025 to 2026 ones.
- **Why it matters:** the speed tail and the old-vehicle tail are the extreme points that will pull on the scaling and on K-Means.

### Standardize the data
K-Means uses Euclidean distance, so a variable measured on a bigger scale would dominate the clustering. We compared the two methods from class:

* **Min-Max scaling (`sta01`):** squeezes each variable into 0 to 1. The range is set by the single most extreme values (the 65 mph crash and the 2017 vehicle).
* **Z-score standardization (`stamn`):** centres each variable on a mean of 0 with a standard deviation of 1.

Both are **linear** rescalings, so the **shape** of each distribution does not change. The chart below shows each variable before and after scaling.

In [ ]:
# @title
def sta01(x):
    return (x - x.min()) / (x.max() - x.min())

def stamn(x):
    return (x - x.mean()) / x.std()

moving["Speed_MinMax"] = sta01(moving[SPEED])
moving["Year_MinMax"] = sta01(moving[YEAR])
moving["Speed_Z"] = stamn(moving[SPEED])
moving["Year_Z"] = stamn(moving[YEAR])

views = {
    "Raw values": (moving[SPEED], moving[YEAR]),
    "Min-Max (0 to 1)": (moving["Speed_MinMax"], moving["Year_MinMax"]),
    "Z-score (mean 0, sd 1)": (moving["Speed_Z"], moving["Year_Z"]),
}
raw_speed_span = moving[SPEED].max() - moving[SPEED].min()
raw_year_span = moving[YEAR].max() - moving[YEAR].min()

fig = make_subplots(rows=2, cols=3, subplot_titles=list(views), vertical_spacing=0.17, horizontal_spacing=0.06)
for col, (speed, year) in enumerate(views.values(), start=1):
    mph = (speed.max() - speed.min()) / raw_speed_span
    fig.add_trace(go.Histogram(x=speed, xbins=dict(start=speed.min() - 0.5 * mph, size=2 * mph),
                               marker_color=PALETTE[0]), row=1, col=col)

    one_year = (year.max() - year.min()) / raw_year_span
    counts = year.round(6).value_counts().sort_index()
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, width=0.7 * one_year,
                         marker_color=PALETTE[1]), row=2, col=col)

    for row, values in ((1, speed), (2, year)):
        axis = (row - 1) * 3 + col
        suffix = "" if axis == 1 else axis
        fig.add_annotation(xref=f"x{suffix} domain", yref=f"y{suffix} domain", x=0.98, y=0.95,
                           xanchor="right", showarrow=False, font=dict(size=13, color=INK),
                           bgcolor="rgba(255,255,255,0.85)",
                           text=f"mean {values.mean():.2f}<br>sd {values.std():.2f}")

fig.update_yaxes(title_text="<b>Speed</b><br>crashes", row=1, col=1)
fig.update_yaxes(title_text="<b>Model year</b><br>crashes", row=2, col=1)
fig.update_layout(height=680, showlegend=False, bargap=0.05,
                  title="<b>Same shape, different ruler: scaling only changes the axis</b>")
fig.show()

**What this shows**
- Each row keeps **exactly the same shape** in all three columns. Scaling does not fix skew or remove outliers, it only changes the numbers on the axis.
- So the real question is not "which one looks better" but **how much weight each variable gets in the distance K-Means calculates**. That is what the spread (standard deviation) after scaling tells us.

In [ ]:
# @title
spread = pd.DataFrame({
    "Scaling": ["Min-Max", "Min-Max", "Z-score", "Z-score"],
    "Variable": ["Speed", "Model year", "Speed", "Model year"],
    "Spread (sd)": [moving["Speed_MinMax"].std(), moving["Year_MinMax"].std(),
                    moving["Speed_Z"].std(), moving["Year_Z"].std()],
})

fig = px.bar(spread, x="Scaling", y="Spread (sd)", color="Variable", barmode="group", text_auto=".2f",
             color_discrete_sequence=PALETTE[:2])
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(height=430, legend_title_text="",
                  title="<b>How much each variable counts in the K-Means distance</b>",
                  yaxis=dict(title="Standard deviation after scaling", range=[0, 1.2]))
fig.show()

**Decision: Z-score standardization.**
- With **Min-Max**, the spread of each variable is set by its single most extreme value, so speed and model year end up with **unequal weight** (sd 0.17 vs 0.14). That weighting is an accident of where the outliers happen to sit.
- With **Z-score**, both variables have a standard deviation of exactly 1, so each gets an **equal say** in the distance.
- **What Z-score does *not* do:** it is not outlier-robust, because it uses the mean and standard deviation, which outliers pull on. The 2017 to 2022 vehicles end up 1.6 to 5.6 standard deviations below the mean, far from the 2024 mass. Keep this in mind: it is part of why the older vehicles form their own cluster later.

We continue with the Z-scored variables.

## ④ Run K-Means for k = 1 to 7
We run K-Means for every $k$ from 1 to 7 and track the Sum of Squares (SS) metrics from class:
* **Total SS:** the total variance in the data. It is the same for every $k$.
* **Within SS (inertia):** variance *inside* the clusters. Lower is better.
* **Between SS:** variance *between* the cluster centres. Higher is better.
* **Between / Total:** the share of the variance the clusters explain.

In [ ]:
# @title
X = moving[["Speed_Z", "Year_Z"]]
total_ss = ((X - X.mean()) ** 2).sum().sum()

rows, labels = [], {}
for k in K_RANGE:
    model = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X)
    labels[k] = model.labels_
    rows.append({"k": k, "Total SS": total_ss, "Within SS": model.inertia_, "Between SS": total_ss - model.inertia_})

ss = pd.DataFrame(rows)
ss["Between / Total"] = ss["Between SS"] / ss["Total SS"]
ss.style.format({"Total SS": "{:.2f}", "Within SS": "{:.2f}", "Between SS": "{:.2f}",
                 "Between / Total": "{:.0%}"}).hide(axis="index")

k,Total SS,Within SS,Between SS,Between / Total
1,1132.00,1132.00,0.00,0%
2,1132.00,737.62,394.38,35%
3,1132.00,378.05,753.95,67%
4,1132.00,277.79,854.21,75%
5,1132.00,209.33,922.67,82%
6,1132.00,170.16,961.84,85%
7,1132.00,139.00,993.00,88%


## ⑤ Choose k
### Elbow chart
The elbow chart plots **Within SS** against $k$. Within SS always falls as $k$ grows, so we look for the "elbow": the point after which adding another cluster stops buying much.

In [ ]:
# @title
fig = go.Figure(go.Scatter(
    x=ss["k"], y=ss["Within SS"], mode="lines+markers+text",
    text=ss["Within SS"].round(0).astype(int), textposition="top right",
    line=dict(color=PALETTE[0], width=3), marker=dict(size=11)))
fig.add_vline(x=FINAL_K, line_dash="dash", line_color=PALETTE[1], line_width=2,
              annotation_text=f"<b>Elbow: k = {FINAL_K}</b>", annotation_position="top right",
              annotation_font=dict(color=PALETTE[1], size=15))
fig.update_layout(height=470, showlegend=False,
                  title=f"<b>The gain flattens after k = {FINAL_K}</b>",
                  xaxis=dict(title="Number of clusters (k)", dtick=1),
                  yaxis=dict(title="Within SS (inertia)", range=[0, total_ss * 1.12]))
fig.show()

**Justification for k = 3**
- Going from k = 1 to 2 removes **394** units of Within SS, and k = 2 to 3 removes another **360**.
- After that the gains collapse: k = 3 to 4 removes only **100**, and k = 4 to 5 only **68**.
- At **k = 3** the clusters already explain **67%** of the total variance.
- Following the principle of parsimony, **k = 3** gives clear, meaningful groups without over-splitting the data into clusters we could not tell apart.

### Interactive cluster exploration
**Drag the slider** under the chart to see how the crashes are split as $k$ goes from 2 to 7. Cluster numbers (and so colours) are arbitrary and can change between values of $k$. Hover over a point to see its real speed and model year.

In [ ]:
# @title
slider_df = moving[["Speed_Z", "Year_Z"]].assign(speed_mph=moving[SPEED], model_year=moving[YEAR])
for k in range(2, MAX_K + 1):
    slider_df[f"k{k}"] = labels[k].astype(str)

k_param = alt.param(name="k_val", value=FINAL_K,
                    bind=alt.binding_range(min=2, max=MAX_K, step=1, name="Number of clusters (k): "))

alt.Chart(slider_df).mark_circle(size=70, opacity=0.7).encode(
    x=alt.X("Speed_Z:Q", title="Speed (z-score)"),
    y=alt.Y("Year_Z:Q", title="Model year (z-score)"),
    color=alt.Color("cluster:N", title="Cluster", scale=alt.Scale(range=PALETTE)),
    tooltip=[alt.Tooltip("speed_mph:Q", title="Speed (mph)"),
             alt.Tooltip("model_year:Q", title="Model year", format="d"),
             alt.Tooltip("cluster:N", title="Cluster")],
).transform_calculate(
    cluster='datum["k" + k_val]'
).add_params(
    k_param
).properties(
    title=alt.TitleParams("K-Means clusters as k changes", subtitle="Drag the slider below the chart"),
    width=620, height=420,
).configure_title(
    fontSize=20, anchor="start", font="Inter, Segoe UI, Arial, sans-serif"
).interactive()

alt.Chart(...)

## ⑥ Name and explain the clusters
### Final clusters: k = 3
We fit K-Means once more with **k = 3**, then name each cluster from its average speed and model year. Points are slightly jittered on the chart so crashes with identical values don't hide behind each other; hover shows the real values.

In [ ]:
# @title
km = KMeans(n_clusters=FINAL_K, random_state=SEED, n_init=10).fit(X)
moving["Cluster"] = km.labels_

centres = moving.groupby("Cluster")[[SPEED, YEAR]].mean()
older = centres[YEAR].idxmin()
fast = centres.drop(older)[SPEED].idxmax()
slow = centres.drop([older, fast]).index[0]
names = {fast: "Fast-moving crashes", slow: "Slow-moving crashes", older: "Older-model crashes"}
moving["Cluster name"] = moving["Cluster"].map(names)

ORDER = ["Fast-moving crashes", "Slow-moving crashes", "Older-model crashes"]
COLORS = dict(zip(ORDER, PALETTE))

rng = np.random.default_rng(SEED)
plot_df = moving.assign(jx=moving[SPEED] + rng.uniform(-0.35, 0.35, len(moving)),
                        jy=moving[YEAR] + rng.uniform(-0.3, 0.3, len(moving)))

fig = px.scatter(plot_df, x="jx", y="jy", color="Cluster name", opacity=0.6,
                 category_orders={"Cluster name": ORDER}, color_discrete_map=COLORS,
                 hover_data={"jx": False, "jy": False, SPEED: True, YEAR: True},
                 labels={"jx": "Speed (mph)", "jy": "Model year"},
                 title="<b>Three kinds of moving RoboTaxi crash</b>")
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color="white")))
fig.update_layout(height=520, yaxis=dict(dtick=1), legend_title_text="")
fig.show()

sizes = (moving.groupby("Cluster name")
         .agg(Crashes=(SPEED, "size"), Avg_speed=(SPEED, "mean"), Min_speed=(SPEED, "min"),
              Max_speed=(SPEED, "max"), Avg_model_year=(YEAR, "mean"))
         .reindex(ORDER))
sizes["Share"] = (sizes["Crashes"] / len(moving)).map("{:.0%}".format)
sizes.round(1)

,Crashes,Avg_speed,Min_speed,Max_speed,Avg_model_year,Share
Cluster name,,,,,,
Fast-moving crashes,165,25.30,16,65,2024.30,29%
Slow-moving crashes,360,5.80,1,15,2024.30,63%
Older-model crashes,42,11.50,1,36,2020.40,7%


**What the chart shows**
- **Speed separates two big groups.** *Slow-moving crashes* (1 to 15 mph, average 5.8) and *Fast-moving crashes* (16 to 65 mph, average 25.3). Both have the same average model year (about 2024), so they differ by **how fast the RoboTaxi was going**, not by vehicle age.
- **Model year isolates a small third group.** *Older-model crashes* are all 2017 to 2022 vehicles (average 2020) across a wide speed range (1 to 36 mph, average 11.5).
- **Sizes:** slow = 360 crashes (63%), fast = 165 (29%), older-model = 42 (7%).

### Cluster profiles
The clusters were built **only** from speed and model year. Now we use the **other columns** to describe them. First, the **most common value** of each column inside each cluster, with its share.

In [ ]:
# @title
def top_value(s):
    share = s.value_counts(normalize=True).head(1)
    return f"{share.index[0]} ({share.iloc[0]:.0%})"

moving.groupby("Cluster name")[COLS].agg(top_value).T[ORDER]

Cluster name,Fast-moving crashes,Slow-moving crashes,Older-model crashes
State,TX (36%),CA (47%),CA (57%)
City,Austin (19%),San Francisco (19%),San Francisco (40%)
Roadway Type,Street (70%),Street (61%),Street (71%)
Crash With,Passenger Car (43%),Passenger Car (32%),SUV (38%)
SV Pre-Crash Movement,Proceeding Straight (85%),Proceeding Straight (62%),Proceeding Straight (76%)
CP Pre-Crash Movement,Changing Lanes (25%),Proceeding Straight (41%),Proceeding Straight (29%)


Next, what makes each cluster **different**? For each cluster we list the 3 values that are most **over-represented** compared with all 567 moving crashes (values need at least 5 crashes in the cluster).

In [ ]:
# @title
rows = []
for name, g in moving.groupby("Cluster name"):
    for col in COLS:
        n = g[col].value_counts()
        n = n[n >= 5]
        inside = 100 * n / g[col].notna().sum()
        overall = 100 * moving[col].value_counts(normalize=True)[n.index]
        for value in n.index:
            rows.append([name, col, value, inside[value], overall[value], inside[value] - overall[value]])

insights = pd.DataFrame(rows, columns=["Cluster", "Column", "Value", "% inside cluster", "% of all crashes", "Difference"])
insights = (insights.sort_values("Difference", ascending=False).groupby("Cluster").head(3)
            .sort_values(["Cluster", "Difference"], ascending=[True, False]))
insights.style.format({"% inside cluster": "{:.0f}%", "% of all crashes": "{:.0f}%", "Difference": "+{:.0f}"}).hide(axis="index")

Cluster,Column,Value,% inside cluster,% of all crashes,Difference
Fast-moving crashes,SV Pre-Crash Movement,Proceeding Straight,85%,70%,+15
Fast-moving crashes,CP Pre-Crash Movement,Changing Lanes,25%,12%,+12
Fast-moving crashes,Crash With,Passenger Car,43%,34%,+9
Older-model crashes,City,San Francisco,40%,17%,+23
Older-model crashes,State,CA,57%,42%,+15
Older-model crashes,State,FL,14%,3%,+11
Slow-moving crashes,CP Pre-Crash Movement,Proceeding Straight,41%,32%,+9
Slow-moving crashes,Roadway Type,Parking Lot,13%,8%,+4
Slow-moving crashes,State,CA,47%,42%,+4


Finally, **where** each kind of crash happens. Road type was not used to build the clusters, so any difference here is something the clusters revealed on their own.

In [ ]:
# @title
road_counts = pd.crosstab(moving["Cluster name"], moving["Roadway Type"]).reindex(ORDER)
road = (100 * road_counts.div(road_counts.sum(axis=1), axis=0)).reset_index().melt(
    id_vars="Cluster name", var_name="Roadway type", value_name="Share")

fig = px.bar(road, x="Share", y="Cluster name", color="Roadway type", orientation="h",
             category_orders={"Cluster name": ORDER}, color_discrete_sequence=PALETTE)
fig.update_traces(texttemplate="%{x:.0f}%", textposition="inside", insidetextanchor="middle")
fig.update_layout(height=380, uniformtext_minsize=12, uniformtext_mode="hide", barmode="stack",
                  title="<b>Where each kind of crash happens</b>",
                  xaxis_title="% of crashes in the cluster", yaxis_title="", legend_title_text="")
fig.update_yaxes(autorange="reversed")
fig.show()

print("Parking-lot crashes per cluster:", road_counts.get("Parking Lot", pd.Series(dtype=int)).to_dict())

Parking-lot crashes per cluster: {'Fast-moving crashes': 0, 'Slow-moving crashes': 46, 'Older-model crashes': 1}


**What the tables and chart tell us**
- **Fast-moving crashes:** the RoboTaxi is far more likely to be *proceeding straight* (85% vs 70% overall), and the other vehicle is often *changing lanes* (25% vs 12%) or *making a left turn* (18% vs 10%), so it cuts across the RoboTaxi's path. **None** of them happened in a parking lot.
- **Slow-moving crashes:** almost every parking-lot crash lands here (46 of the 47), giving this cluster a parking-lot share of 13% vs 8% overall. Still, most slow crashes (61%) happen on ordinary streets, so this group is mainly **low-speed driving in traffic**, with parking lots as a distinctive extra.
- **Older-model crashes:** a strongly **San Francisco** group (40% vs 17% overall), with California (57% vs 42%) and Florida (14% vs 3%) also over-represented, and the other vehicle is more often an **SUV** (38% vs 27%). This likely reflects which cities the older fleets were operating in, not proof that older vehicles drive differently.
- **Remember:** these columns *describe* the clusters but were **not used to build them**.

## 🎯 Takeaway
**Moving RoboTaxi crashes fall into three natural groups, separated mainly by speed:**
- **Slow-moving crashes (63%)**, mostly low-speed driving on streets, and home to almost every parking-lot crash.
- **Fast-moving crashes (29%)**, mostly the RoboTaxi going straight while another vehicle changes lanes, and never in a parking lot.
- **A small older-model group (7%)** of 2017 to 2022 vehicles, mostly from San Francisco, with California and Florida over-represented.